# 03 - Explainability with Grad-CAM
Grad-CAM (Selvaraju et al., 2017) highlights the image regions that most increased the
predicted class score. For this system it answers two questions:

1. **Is the model using the right evidence?** Identification of bumble bees depends on the
   colour banding of the thorax and abdomen. A model that looks at the *flower* or *background*
   is exploiting dataset shortcuts (e.g. a species photographed mostly on one plant).
2. **Can an expert trust a suggestion?** Showing the heatmap next to the suggested ID lets
   reviewers quickly see what the model "looked at".

In [ ]:
# --- Environment setup: works on Google Colab and locally -------------------------------
import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/MaryCassieOdie/MSDS688-Case-Study.git"

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = "/content/MSDS688-Case-Study"
    if not os.path.exists(REPO_DIR):
        # If the repo is private, upload the project folder to Drive instead and set
        # REPO_DIR = "/content/drive/MyDrive/MSDS688-Case-Study"
        subprocess.run(["git", "clone", "-q", REPO_URL, REPO_DIR], check=True)
    else:
        subprocess.run(["git", "-C", REPO_DIR, "pull", "-q"], check=False)
    # Data and outputs live on Drive so they survive Colab runtime resets.
    DRIVE_ROOT = "/content/drive/MyDrive/MSDS688-Case-Study"
    os.environ["PROJECT_DATA_DIR"] = f"{DRIVE_ROOT}/data"
    os.environ["PROJECT_OUTPUT_DIR"] = f"{DRIVE_ROOT}/outputs"
else:
    REPO_DIR = os.path.abspath("..")

sys.path.insert(0, os.path.join(REPO_DIR, "src"))

# --- LITE_MODE: quick CPU test of the whole pipeline -------------------------------------
# Lite = 30 images/species at 160 px, 3+2 epochs, small API samples; runs in ~10 min on a
# laptop CPU. Same code paths as the full run, but the numbers are NOT meaningful for the
# write-up. Lite data/outputs go to separate *_lite folders, so they never mix with a full run.
#   None  -> auto: lite when running locally, full on Colab
#   True / False -> force. Restart the kernel after changing it.
LITE_MODE = None
if LITE_MODE is None:
    LITE_MODE = os.environ.get("POLLINATOR_LITE", "0" if IN_COLAB else "1") in ("1", "true", "True")
os.environ["POLLINATOR_LITE"] = "1" if LITE_MODE else "0"
# Optional: official IUCN Red List API token (otherwise iNaturalist's IUCN mirror is used)
# os.environ["IUCN_API_TOKEN"] = "..."   # on Colab, prefer: from google.colab import userdata

from pollinator_id import config
config.ensure_dirs()
print("MODE   ->", "LITE (CPU smoke test - results not meaningful)" if config.LITE else "FULL")
print("data   ->", config.DATA_DIR)
print("output ->", config.OUTPUT_DIR)

In [ ]:
import json
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from pollinator_id import dataset, models, gradcam
from pollinator_id.config import CLASS_NAMES

meta = pd.read_csv(config.METADATA_CSV)
test = meta[meta.split == "test"].reset_index(drop=True)
iucn_codes = dict(pd.read_csv(config.IUCN_CSV)[["scientific_name", "iucn_code"]].values)
best = json.loads((config.OUTPUT_DIR / "best_model.json").read_text())["backbone"]
loaded = {n: models.load_model(n) for n in models.BACKBONES if (config.MODEL_DIR / f"{n}.keras").exists()}
print("best:", best, "| available:", list(loaded))

def raw(fp):
    return dataset.load_image(str(config.IMAGE_DIR / fp)).numpy()

## Correct predictions on threatened species (best model)

In [ ]:
preds = np.load(config.MODEL_DIR / f"{best}_test_predictions.npz")
test["pred"] = [CLASS_NAMES[i] for i in preds["probs"].argmax(1)]
test["conf"] = preds["probs"].max(1)
# make_dataset keeps the order of the test rows (no shuffle outside "train"), so rows align.
assert (test.label.map(CLASS_NAMES.index).to_numpy() == preds["y_true"]).all()

threatened = [n for n in CLASS_NAMES if iucn_codes[n] in {"CR", "EN", "VU"}]
picks = (test[(test.label == test.pred) & test.label.isin(threatened)]
         .sort_values("conf", ascending=False).drop_duplicates("label"))
fig = gradcam.plot_gallery(loaded[best], best, [raw(f) for f in picks.filepath], picks.label.tolist(),
                           save_path=config.FIGURE_DIR / f"gradcam_threatened_{best}.png")

## Misclassifications - what went wrong?

In [ ]:
wrong = test[test.label != test.pred].sort_values("conf", ascending=False).head(6)
print(wrong[["label", "pred", "conf"]].to_string())
if len(wrong):
    fig = gradcam.plot_gallery(loaded[best], best, [raw(f) for f in wrong.filepath], wrong.label.tolist(),
                               save_path=config.FIGURE_DIR / f"gradcam_errors_{best}.png")

## Same images, three backbones

In [ ]:
sample = test.groupby("label").sample(1, random_state=7).head(5)
imgs = [raw(f) for f in sample.filepath]
fig, axes = plt.subplots(len(loaded) + 1, len(imgs), figsize=(2.7 * len(imgs), 2.8 * (len(loaded) + 1)))
for j, (img, true) in enumerate(zip(imgs, sample.label)):
    axes[0, j].imshow(img.astype("uint8")); axes[0, j].set_title(true, fontsize=8)
    for i, (name, m) in enumerate(loaded.items(), start=1):
        ov, _, idx, p = gradcam.explain(m, name, img)
        axes[i, j].imshow(ov)
        axes[i, j].set_title(f"{name}: {CLASS_NAMES[idx].split()[1]} {p[idx]:.0%}", fontsize=8,
                             color="tab:green" if CLASS_NAMES[idx] == true else "tab:red")
for a in axes.ravel(): a.axis("off")
fig.tight_layout(); fig.savefig(config.FIGURE_DIR / "gradcam_backbone_comparison.png", dpi=130)

## A simple quantitative check: is attention on the insect?
Without bounding boxes we cannot measure this exactly, but iNaturalist photographers usually centre
the subject. We compute the share of Grad-CAM mass inside the central 50% of the image for every
test image. A model relying on background context will score noticeably lower.

In [ ]:
def center_share(hm):
    h, w = hm.shape; t = hm.sum() + 1e-8
    return hm[h // 4: 3 * h // 4, w // 4: 3 * w // 4].sum() / t

rows = []
for name, m in loaded.items():
    for fp in test.sample(min(config.GRADCAM_SAMPLE, len(test)), random_state=0).filepath:
        _, hm, _, _ = gradcam.explain(m, name, raw(fp))
        rows.append({"model": name, "center_share": center_share(hm)})
cs = pd.DataFrame(rows)
ax = cs.boxplot(column="center_share", by="model", figsize=(7, 4)); plt.suptitle("")
ax.axhline(0.25, ls="--", c="gray"); ax.set_title("Grad-CAM mass in central 50% of image (dashed = uniform)")
cs.groupby("model").center_share.describe().round(3)

### Discussion prompts
* Do heatmaps concentrate on the abdomen/thorax banding that field guides use?
* For errors: did the model look at the right body part but confuse similar patterns, or did it
  look at the flower/background (a data problem)?
* Grad-CAM uses the last conv layer (7x7 for all three backbones at 224 px), so maps are coarse.